We use the advanced search on pdb to look for:


    Identifier = "PF00014"
            AND
            Annotation Type = "Pfam"
      AND
      Data Collection Resolution <= 3.5
      AND
      Polymer Entity Sequence Length > 40
      AND
      Polymer Entity Sequence Length < 80

We that do a custom report and extract the information:
- Sequence
- Auth Asym ID
- Entity ID
The pdb id is automatically added

We download in csv format then we filter our pandas dataframe

In [10]:
import pandas as pd

df = pd.read_csv('pdb_custom_report.csv')
df
#We read the CSV file into a DataFrame and display its contents to check the initial structure and data.

,Identifier,Polymer EntityData,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,Entry ID,Sequence,Auth Asym ID,Entity ID,NaN
1,1AAL,RPDFCLEPPYTGPCKARIIRYFYNAKAGLVQTFVYGGCRAKRNNFK...,B,1,NaN
2,NaN,NaN,A,NaN,NaN
3,1AAP,VREVCSEQAETGPCRAMISRWYFDVTEGKCAPFFYGGCGGNRNNFD...,A,1,NaN
4,NaN,NaN,B,NaN,NaN
...,...,...,...,...,...
408,NaN,NaN,B,NaN,NaN
409,NaN,CGVPAIQPVLSGL,E,1,NaN
410,NaN,NaN,A,NaN,NaN
411,8PTI,RPDFCLEPPYTGPCKARIIRYFYNAKAGLCQTFVGGGCRAKRNNFK...,A,1,NaN


In [11]:
df.columns = df.iloc[0]
df = df[1:].copy()
#To set the second row as header and fix the formatting of the DataFrame

In [12]:
df = df[df["Sequence"].notna()]
df = df[df["Sequence"].astype(str).str.strip() != ""]
df = df[df["Sequence"].astype(str).str.len().between(40, 80)]
df = df.dropna(subset=["Entry ID"])
df.shape
df
#To filter the DataFrame based on the specified criteria and check the resulting shape and contents

,Entry ID,Sequence,Auth Asym ID,Entity ID,NaN
1,1AAL,RPDFCLEPPYTGPCKARIIRYFYNAKAGLVQTFVYGGCRAKRNNFK...,B,1,NaN
3,1AAP,VREVCSEQAETGPCRAMISRWYFDVTEGKCAPFFYGGCGGNRNNFD...,A,1,NaN
5,1B0C,RPDFCLEPPYTGPCKARIIRYFYNAKAGLCQTFVYGGCRAKRNNFK...,B,1,NaN
10,1BHC,RPDFCLEPPYTGPCKARIIRYFYNAKAGLCQTFVYGGCRAKRNNFK...,A,1,NaN
20,1BPI,RPDFCLEPPYTGPCKARIIRYFYNAKAGLCQTFVYGGCRAKRNNFK...,A,1,NaN
...,...,...,...,...,...
387,7QIR,RPDFCLEPPYTGPCXARIIRYFYNAKAGLCQTFVYGGCRAKRNNFK...,D,4,NaN
395,7QIS,RPDFCLEPPYTGPCXARIIRYFYNAKAGLCQTFVYGGCRAKRNNFK...,H,4,NaN
403,7QIT,RPDFCLEPPYTGPCXARIIRYFYNAKAGLCQTFVYGGCRAKRNNFK...,H,4,NaN
411,8PTI,RPDFCLEPPYTGPCKARIIRYFYNAKAGLCQTFVGGGCRAKRNNFK...,A,1,NaN


In [13]:
with open("kunitz_candidates.fasta", "w") as f:
    for _, row in df.iterrows():
        header = f">{row['Entry ID']}_{row['Auth Asym ID']}"
        sequence = str(row["Sequence"]).replace(" ", "").replace("\n", "")
        f.write(header + "\n")
        f.write(sequence + "\n")

#We make the file we will inpit into MMseqs2, with the header containing the Entry ID and Asym ID, and the sequence cleaned of any spaces or newlines.        

We input the created file into the online tool MMseq2 and download the provided file of the clusters (We could fix the parameters of Minimum sequence identity and Minimum alignment coverage into 0.95)

In [14]:
#Our output file was mmseqs2_4251780.out
ids = []
chains = []

with open("mmseqs2_4251780.out", "r") as f:
    for line in f:
        line = line.strip()
        if line and line.startswith(">"):
            ids.append(line[1:].split("_")[0])
            chains.append(line[1:].split("_")[1])

with open("clusters.txt", "w") as f:
    for id, chain in zip(ids, chains):
        f.write(f"{id}:{chain}\n")
#We extract the Entry IDs and chain IDs from the MMseqs2 output file and write them to a new file in the specified format for input in pdbefold  


We use the online tool of PDBeFold for the structual alignment. We chose multiple alignment and input the file clusters.txt we created and view the results. 
We download the FASTA alignment optionaly the Rasmol file to view the superimposition on Chimera

For the HMM

We download our positive and negative sample
From uniprot we use the advanced parameter search to download two seperate databases we will use to train our model. One is the positive for kunitz and the other is the negative which normaly is a much larger sample. 